# Chapter 2 — Function Calling and Structured Outputs · Homework (self-test)

**Goal:** prove you can build, steer and validate OpenAI function calling and Structured Outputs without looking at the answers.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 2–3 hours
- **Estimated API cost:** $0 (everything is offline; the one optional bonus cell marked **(live)** costs < $0.01 with `gpt-6-luna`)
- **Prerequisites:** the practice notebook

### How to use this notebook

1. Attempt everything **without peeking**. Solutions are at the very bottom.
2. **Part A** — 10 scenario questions: write your letters in the `answers` dict and run the grader. The answer key is stored as hashes, so you can't read it by accident.
3. **Part B** — 6 coding exercises: replace each `TODO` stub, then run its `check_exN()` cell. Checkers never crash the notebook; they print a hint when something is off.
4. **Part C** — one architecture scenario, graded by yourself against a rubric.
5. **Scorecard** — the pass mark is **72%** of quiz + exercises (the same bar as the Claude path, which mirrors the Claude exam's 720/1000).

## Setup

In [ ]:
import hashlib
import json
import os
from typing import Optional

import openai
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(find_dotenv())
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")
print("openai SDK", openai.__version__, "| model:", MODEL)

Only the optional bonus cell needs a key. Skip this cell if you don't have one; everything else is offline.

In [ ]:
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env (see 00-prerequisites)"
client = OpenAI()

Offline helpers (the same ones as in the practice notebook) and shared definitions:

In [ ]:
from openai.types.responses import Response


def fake_response(*items: dict, status: str = "completed", incomplete_reason: str | None = None,
                  rid: str = "resp_fake") -> Response:
    """Build a real SDK `Response` object from plain dicts, so offline code sees the same types as live code."""
    return Response.model_validate({
        "id": rid, "object": "response", "created_at": 0, "model": "offline-fake",
        "status": status,
        "incomplete_details": {"reason": incomplete_reason} if incomplete_reason else None,
        "parallel_tool_calls": True, "tool_choice": "auto", "tools": [],
        "output": list(items),
    })


def fc(name: str, args: dict, call_id: str) -> dict:
    """A `function_call` output item. Note: `arguments` is a JSON *string*."""
    return {"type": "function_call", "id": "fc_" + call_id.split("_")[-1], "call_id": call_id,
            "name": name, "arguments": json.dumps(args), "status": "completed"}


def msg(text: str, phase: str | None = None) -> dict:
    """An assistant `message` output item with one `output_text` part (optionally with a `phase`)."""
    item = {"type": "message", "id": "msg_fake", "role": "assistant", "status": "completed",
            "content": [{"type": "output_text", "text": text, "annotations": []}]}
    if phase is not None:
        item["phase"] = phase          # "commentary" (preamble) or "final_answer"
    return item


def refusal(text: str) -> dict:
    """An assistant `message` whose content is a `refusal` part instead of `output_text`."""
    return {"type": "message", "id": "msg_refusal", "role": "assistant", "status": "completed",
            "content": [{"type": "refusal", "refusal": text}]}


def reasoning(rid: str = "rs_fake") -> dict:
    """A (summary-less) reasoning item, as reasoning models return alongside tool calls."""
    return {"type": "reasoning", "id": rid, "summary": []}


class ScriptedClient:
    """Stands in for `OpenAI()`: `client.responses.create(**kw)` returns pre-scripted responses
    in order and records a snapshot of every request, so loops can run (and be checked) offline."""

    def __init__(self, scripted: list[Response]):
        self._queue = list(scripted)
        self.requests: list[dict] = []
        self.responses = self          # so `client.responses.create(...)` works

    def create(self, **kwargs) -> Response:
        snapshot = dict(kwargs)
        if isinstance(snapshot.get("input"), list):
            snapshot["input"] = list(snapshot["input"])   # copy: the caller keeps mutating its list
        self.requests.append(snapshot)
        if not self._queue:
            raise RuntimeError("ScriptedClient: no scripted responses left (loop did not stop?)")
        return self._queue.pop(0)


def item_type(x) -> str | None:
    """Type of an input/output item, whether it is an SDK object or a plain dict."""
    return x.get("type") if isinstance(x, dict) else getattr(x, "type", None)

In [ ]:
# Fake in-memory "backend" for the customer-support examples. No real systems are touched.
CUSTOMERS = {
    "C-001": {"customer_id": "C-001", "name": "Jane Doe", "email": "jane@example.com",
              "status": "active", "order_ids": ["A-1001", "A-1002"]},
    "C-002": {"customer_id": "C-002", "name": "Sam Lee", "email": "sam@example.com",
              "status": "active", "order_ids": ["A-2001"]},
}
ORDERS = {
    "A-1001": {"order_id": "A-1001", "customer_id": "C-001", "status": "delivered",
               "total_usd": 89.90, "items": ["Wireless mouse", "USB-C cable"]},
    "A-1002": {"order_id": "A-1002", "customer_id": "C-001", "status": "shipped",
               "total_usd": 249.00, "items": ["Mechanical keyboard"]},
    "A-2001": {"order_id": "A-2001", "customer_id": "C-002", "status": "delivered",
               "total_usd": 35.50, "items": ["Phone case"]},
}


get_customer_tool = {
    "type": "function",
    "name": "get_customer",
    "description": (
        "Look up ONE customer by email or by customer ID and return name, account status and "
        "their order IDs (not order details). Call this FIRST to verify who you are talking to, "
        "before any refund. Pass exactly one of email / customer_id and set the other to null."
    ),
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": ["string", "null"], "description": "Customer email, e.g. jane@example.com, or null"},
            "customer_id": {"type": ["string", "null"], "description": "Customer ID like C-001, or null"},
        },
        "required": ["email", "customer_id"],
        "additionalProperties": False,
    },
}

lookup_order_tool = {
    "type": "function",
    "name": "lookup_order",
    "description": (
        "Look up ONE order by order ID and return its status, items, total in USD and owning "
        "customer ID. Use it for any question about an order's status or contents. Do NOT use it "
        "to identify a customer. Order IDs look like A-1001."
    ),
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {"order_id": {"type": "string", "description": "Order ID, e.g. A-1001"}},
        "required": ["order_id"],
        "additionalProperties": False,
    },
}

process_refund_tool = {
    "type": "function",
    "name": "process_refund",
    "description": (
        "Refund all or part of a DELIVERED order that belongs to the verified customer. "
        "Irreversible. Only call after get_customer and lookup_order, and only for the amount "
        "the customer asked for (never more than the amount still refundable on the order). Refunds above 100 USD are "
        "not executed; they return requires_human_approval."
    ),
    "strict": True,
    "parameters": {
        "type": "object",
        "properties": {
            "order_id": {"type": "string", "description": "Order ID, e.g. A-1001"},
            "amount_usd": {"type": "number", "minimum": 0, "description": "Refund amount in USD"},
            "reason": {"type": "string", "enum": ["damaged", "not_as_described", "late", "other"]},
        },
        "required": ["order_id", "amount_usd", "reason"],
        "additionalProperties": False,
    },
}

SUPPORT_TOOLS = [get_customer_tool, lookup_order_tool, process_refund_tool]


import copy

SUPPORT_INSTRUCTIONS = (
    "You are a customer-support agent. Verify the customer with get_customer before refunds. "
    "Use tools for facts; never invent order data."
)

class LineItem(BaseModel):
    description: str
    amount_usd: float

class InvoiceExtraction(BaseModel):
    invoice_number: Optional[str]
    line_items: list[LineItem]
    stated_total_usd: float

RESULTS: dict[str, bool] = {}   # filled in by the check_exN() cells

def _report(ex: str, fn) -> None:
    """Run a checker; record pass/fail; never let an exception escape."""
    try:
        fn()
        RESULTS[ex] = True
        print(f"✓ Exercise {ex[2:]} passed")
    except NotImplementedError:
        RESULTS[ex] = False
        print(f"✗ Exercise {ex[2:]}: not implemented yet")
    except AssertionError as exc:
        RESULTS[ex] = False
        print(f"✗ Exercise {ex[2:]}: {exc}")
    except Exception as exc:
        RESULTS[ex] = False
        print(f"✗ Exercise {ex[2:]}: your code raised {type(exc).__name__}: {exc}")

print("ready:", [t["name"] for t in SUPPORT_TOOLS])

---
## Part A — Quiz (10 questions)

One correct answer per question. [Theory](README.md#concepts)

**Q1. (Round trip)** The model returned a `function_call` item `item`. Which input item correctly sends your function's result back in the next Responses request?

- A) `{"role": "tool", "tool_call_id": item.id, "content": result}`
- B) `{"type": "function_call_output", "call_id": item.id, "output": result}`
- C) `{"type": "function_call_output", "call_id": item.call_id, "output": result}`
- D) `{"type": "tool_result", "tool_use_id": item.call_id, "content": result}`

**Q2. (Arguments)** Your handler does `order_id = item.arguments["order_id"]` and crashes with `TypeError: string indices must be integers`. What is the fix?

- A) Turn on `strict: true` so the arguments come back as an object
- B) `json.loads(item.arguments)["order_id"]`, because `arguments` is a JSON-encoded string
- C) Read `item.input["order_id"]` instead
- D) Set `parallel_tool_calls: false`

**Q3. (Strict schema)** A strict `get_customer` tool lists `email` and `customer_id` in `properties` but only `email` in `required`, because callers send one or the other. The request is rejected. What is the correct fix?

- A) List both in `required` and type both as `["string", "null"]`, describing that the unused one is null
- B) Remove `additionalProperties: false` so the schema is looser
- C) Drop `strict` and validate in code
- D) Add `"default": null` to `customer_id`

**Q4. (Strict default)** A team never sets `strict` on its Responses function tools. One tool's schema leaves a property out of `required`, and that tool occasionally receives arguments with the wrong type. What explains it?

- A) Responses ignores schemas unless `strict` is set
- B) Responses always behaves like Chat Completions, which is non-strict by default
- C) The model doesn't support function calling
- D) Responses tried to normalize the schema into strict mode, couldn't, and fell back to non-strict best-effort calling (the tool shows `strict: false`)

**Q5. (tool_choice)** Documents arrive as invoices, receipts or contracts, and you have one extraction function per type. You must always get a function call (never prose) but the model should pick the function. On Claude you used `{"type": "any"}`. What is the OpenAI equivalent?

- A) `tool_choice: "auto"`
- B) `tool_choice: "required"`
- C) `tool_choice: {"type": "function", "name": "extract_invoice"}`
- D) `tool_choice: "none"`

**Q6. (allowed_tools)** Your agent sends the same three tools on every turn so the prompt prefix stays cacheable. Before identity verification, `process_refund` must not be callable. Which setting fits best?

- A) Add "do not refund before verification" to `instructions`
- B) Remove `process_refund` from `tools` until verification
- C) `tool_choice: {"type": "allowed_tools", "mode": "auto", "tools": [get_customer, lookup_order references]}`
- D) `max_tool_calls: 0` until verification

**Q7. (Parallel calls)** The second call in your workflow needs an ID returned by the first, and the model keeps emitting both calls in one turn with a guessed ID. Which request change ensures at most one call per turn?

- A) `parallel_tool_calls: false`
- B) `tool_choice: "none"`
- C) `tool_choice: {"type": "auto", "disable_parallel_tool_use": true}`
- D) `max_tool_calls: 1`

**Q8. (Refusals)** `client.responses.parse(text_format=Claim)` returns `status == "completed"` but `output_parsed` is `None`. What is the most likely cause?

- A) The schema is invalid; the API would have used JSON mode instead
- B) `max_output_tokens` was reached
- C) The network dropped the body
- D) The model refused: the message contains a `refusal` content part instead of `output_text`

**Q9. (Migration)** You move a Chat Completions call that uses `response_format={"type": "json_schema", "json_schema": {...}}` to the Responses API. What is the correct Responses form?

- A) Keep `response_format` unchanged
- B) `text={"format": {"type": "json_schema", "name": ..., "strict": True, "schema": ...}}`
- C) Pass both `text={"format": ...}` and `text_format=Model` to `parse()` for safety
- D) `text={"format": {"type": "json_object"}}`

**Q10. (Tool errors)** `lookup_order` receives an order ID that doesn't exist. What should your loop send back?

- A) A `function_call_output` for that `call_id` whose `output` is a JSON error such as `{"ok": false, "error": "not_found", "retryable": false, "message": "...ask the user to check the ID"}`
- B) Nothing for that call; only send outputs for successful calls
- C) A `function_call_output` with `"is_error": true`
- D) Raise the exception so the whole request is retried

In [ ]:
answers = {
    "q1": None,  # "A" / "B" / "C" / "D"
    "q2": None,  # "A" / "B" / "C" / "D"
    "q3": None,  # "A" / "B" / "C" / "D"
    "q4": None,  # "A" / "B" / "C" / "D"
    "q5": None,  # "A" / "B" / "C" / "D"
    "q6": None,  # "A" / "B" / "C" / "D"
    "q7": None,  # "A" / "B" / "C" / "D"
    "q8": None,  # "A" / "B" / "C" / "D"
    "q9": None,  # "A" / "B" / "C" / "D"
    "q10": None,  # "A" / "B" / "C" / "D"
}

In [ ]:
ANSWER_KEY = {
    "q1": "227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e",
    "q2": "61e5302385b2dd8204dbddfd86b357cacb0de07dbc800460aa335d9bc96b6ca1",
    "q3": "2501e2fe43d858f1773d414ee14ab0a8cdf2afefbc4ffd20ae5d956e618f3ea3",
    "q4": "b634a8333266441f30307b2d3392d77b7b3bcf69bc053cabed2c65d2671fc12e",
    "q5": "fbb43cbbf51767ad8ca178fa6cc4731661f1f1771789b9b7df93fba7ad386e2c",
    "q6": "18cb69e7d9c041e4c1b21fa8c0e110716664b55bef1348d49301f8891c82cef1",
    "q7": "8722338db4eb12eaae5f6f26c68875af2b115581d770f01f98b9e0e1e6010ac1",
    "q8": "85b91a1895880cedf03bb98aa9ebfb3c730c838ec5d8f59ec2d1d86bcb936ced",
    "q9": "b25dce13077091b19d36fd594efa7fa456f167c4a463522f004183e5d5d2bb5e",
    "q10": "aeb5147fa5276b8759290ab8af9ed3aaa01d2c844cecc89ea5547b013e209ad3",
}


def grade_quiz(answers: dict) -> int:
    """Compare against hashed answers; unanswered counts as wrong. Stores the score in QUIZ_SCORE."""
    global QUIZ_SCORE
    score = 0
    for qid, digest in ANSWER_KEY.items():
        letter = (answers.get(qid) or "").strip().upper()
        ok = bool(letter) and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == digest
        score += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    QUIZ_SCORE = score
    print(f"\nQuiz: {score}/{len(ANSWER_KEY)} ({score / len(ANSWER_KEY):.0%})")
    return score

grade_quiz(answers)

---
## Part B — Coding exercises

All exercises run offline against fake `Response` objects built from the real SDK types. [Theory](README.md#concepts)

### Exercise 1 — Convert a Chat Completions tool to the Responses shape

`openai.pydantic_function_tool()` and most older code produce the **nested** Chat Completions shape. Write `to_responses_tool(chat_tool)` that returns the **flat** Responses shape:

- `type` is `"function"`, `name` comes from `function.name`.
- Copy `description`, `parameters` and `strict` **only if present** (don't invent keys).
- Don't mutate the input (deep-copy `parameters`).
- Raise `ValueError` if the input isn't a Chat Completions function tool (no `"function"` key or `type != "function"`).

In [ ]:
def to_responses_tool(chat_tool: dict) -> dict:
    # TODO: return the flat Responses-shaped tool
    raise NotImplementedError

In [ ]:
def check_ex1():
    from pydantic import BaseModel as _BM

    class GetWeather(_BM):
        """Get the current weather for a city."""
        city: str

    nested = openai.pydantic_function_tool(GetWeather, name="get_weather")
    before = copy.deepcopy(nested)
    flat = to_responses_tool(nested)
    assert isinstance(flat, dict), "return a dict"
    assert "function" not in flat, "the result must be flat: no 'function' key"
    assert flat.get("type") == "function" and flat.get("name") == "get_weather", "type/name wrong"
    assert flat.get("strict") is True, "keep strict=True from the source tool"
    assert flat.get("parameters") == nested["function"]["parameters"], "parameters must be copied as-is"
    assert flat.get("description") == "Get the current weather for a city.", "keep the description"
    assert nested == before, "don't mutate the input tool"
    flat["parameters"]["properties"]["city"]["type"] = "integer"
    assert nested == before, "deep-copy parameters (editing the result changed the input)"

    minimal = to_responses_tool({"type": "function", "function": {"name": "ping", "parameters": {"type": "object", "properties": {}}}})
    assert set(minimal) == {"type", "name", "parameters"}, f"only copy keys that exist; got {sorted(minimal)}"

    try:
        to_responses_tool({"type": "function", "name": "already_flat", "parameters": {}})
    except ValueError:
        pass
    else:
        raise AssertionError("a flat (Responses) tool should raise ValueError")

_report("ex1", check_ex1)

### Exercise 2 — Make a schema strict-compliant

Write `make_strict(schema)` that returns a **new** schema satisfying OpenAI strict mode:

- Every object gets `additionalProperties: false` and lists **all** its properties in `required`. That includes nested objects, objects inside `items` of arrays, inside `anyOf` branches, and every schema under `$defs` (strict mode supports definitions and `$ref`).
- Properties that were **not** originally required become nullable. Originally required properties stay as they are.
  - With a `type`: `"string"` → `["string", "null"]`; a list type gets `"null"` appended if missing. If the property also has an `enum`, append `None` (JSON `null`) to the enum too, otherwise the enum still forbids `null`.
  - Without a `type`: an `anyOf` gets a `{"type": "null"}` branch appended (if missing); a `{"$ref": ...}` becomes `{"anyOf": [{"$ref": ...}, {"type": "null"}]}`.
- Don't mutate the input.

In [ ]:
def make_strict(schema: dict) -> dict:
    # TODO: return a strict-compliant deep copy
    raise NotImplementedError

In [ ]:
def check_ex2():
    loose = {
        "type": "object",
        "properties": {
            "email": {"type": "string"},
            "customer_id": {"type": "string"},
            "tags": {"type": "array", "items": {
                "type": "object",
                "properties": {"label": {"type": "string"}, "score": {"type": "number"}},
                "required": ["label"]}},
            "address": {"type": "object",
                        "properties": {"city": {"type": "string"}, "zip": {"type": ["string"]}},
                        "required": ["city"]},
        },
        "required": ["email", "tags"],
    }
    before = copy.deepcopy(loose)
    s = make_strict(loose)
    assert loose == before, "don't mutate the input schema"
    assert s.get("additionalProperties") is False, "root object needs additionalProperties: false"
    assert set(s.get("required", [])) == {"email", "customer_id", "tags", "address"}, "root: every property must be required"
    p = s["properties"]
    assert p["email"]["type"] == "string", "email was required: keep its type as 'string'"
    assert p["customer_id"]["type"] == ["string", "null"], "customer_id was optional: make it ['string', 'null']"
    assert p["tags"]["type"] == "array", "tags was required: keep 'array'"
    item = p["tags"]["items"]
    assert item.get("additionalProperties") is False, "objects inside array items need additionalProperties: false"
    assert set(item.get("required", [])) == {"label", "score"}, "array item object: all properties required"
    assert item["properties"]["score"]["type"] == ["number", "null"], "score was optional: make it nullable"
    addr = p["address"]
    assert addr["type"] == ["object", "null"], "address was optional: its type becomes ['object', 'null']"
    assert addr.get("additionalProperties") is False and set(addr.get("required", [])) == {"city", "zip"}, \
        "nested object: additionalProperties false + all properties required"
    assert addr["properties"]["zip"]["type"] == ["string", "null"], "a list type gets 'null' appended"
    assert make_strict(s) == s, "running make_strict on a strict schema should change nothing"

    # Enums, $ref / anyOf properties and $defs (what Pydantic generates for nested and Optional fields)
    with_defs = {
        "type": "object",
        "properties": {
            "status": {"type": "string", "enum": ["open", "closed"]},
            "owner": {"$ref": "#/$defs/person"},
            "ref": {"anyOf": [{"type": "string"}, {"type": "integer"}]},
            "lead": {"$ref": "#/$defs/person"},
        },
        "required": ["lead"],
        "$defs": {"person": {"type": "object",
                             "properties": {"name": {"type": "string"}, "phone": {"type": "string"}},
                             "required": ["name"]}},
    }
    before = copy.deepcopy(with_defs)
    d = make_strict(with_defs)
    assert with_defs == before, "don't mutate the input schema"
    dp = d["properties"]
    assert dp["status"].get("type") == ["string", "null"] and None in dp["status"].get("enum", []), \
        "optional enum: make the type nullable AND add None (null) to the enum"
    assert dp["owner"] == {"anyOf": [{"$ref": "#/$defs/person"}, {"type": "null"}]}, \
        "optional $ref: wrap it as {'anyOf': [{'$ref': ...}, {'type': 'null'}]}"
    assert dp["ref"].get("anyOf") == [{"type": "string"}, {"type": "integer"}, {"type": "null"}], \
        "optional anyOf: append a {'type': 'null'} branch"
    assert dp["lead"] == {"$ref": "#/$defs/person"}, "a required $ref stays as it is"
    person = d.get("$defs", {}).get("person", {})
    assert person.get("additionalProperties") is False and set(person.get("required", [])) == {"name", "phone"}, \
        "objects under $defs must be made strict too"
    assert person["properties"]["phone"]["type"] == ["string", "null"], "optional fields under $defs become nullable"
    assert make_strict(d) == d, "running make_strict on a strict schema should change nothing"

_report("ex2", check_ex2)

### Exercise 3 — Build a phase-aware support request

Write `build_turn_request(verified, input_list)` that returns the keyword arguments for `client.responses.create(**kwargs)` in the support agent:

- Always: `model=MODEL`, `instructions=SUPPORT_INSTRUCTIONS`, `input=input_list`, and **the full, unchanged** `tools=SUPPORT_TOOLS` (stable prefix for caching).
- Always: at most one tool call per turn (the refund flow must be sequential).
- Not verified: only `get_customer` and `lookup_order` may be called, **and** a call is required.
- Verified: the model decides freely among all tools.

In [ ]:
def build_turn_request(verified: bool, input_list: list) -> dict:
    # TODO: return the kwargs dict for client.responses.create
    raise NotImplementedError

In [ ]:
def check_ex3():
    msgs = [{"role": "user", "content": "refund please"}]
    before = build_turn_request(False, msgs)
    after = build_turn_request(True, msgs)
    for name, req in (("unverified", before), ("verified", after)):
        assert isinstance(req, dict), "return a dict of kwargs"
        assert req.get("model") == MODEL, f"{name}: model must be MODEL"
        assert req.get("instructions") == SUPPORT_INSTRUCTIONS, f"{name}: pass SUPPORT_INSTRUCTIONS"
        assert req.get("input") is msgs, f"{name}: pass input_list as input"
        assert req.get("tools") == SUPPORT_TOOLS, f"{name}: always send the full, unchanged SUPPORT_TOOLS"
        assert req.get("parallel_tool_calls") is False, f"{name}: disable parallel calls (top-level parallel_tool_calls)"
        assert "response_format" not in req, "response_format is Chat Completions; not needed here"
    tc = before.get("tool_choice")
    assert isinstance(tc, dict) and tc.get("type") == "allowed_tools", \
        "unverified: use tool_choice type 'allowed_tools' (don't shrink the tools list)"
    assert tc.get("mode") == "required", "unverified: a call is required -> mode 'required'"
    names = {t.get("name") for t in tc.get("tools", []) if t.get("type") == "function"}
    assert names == {"get_customer", "lookup_order"}, f"unverified: allowed tools should be get_customer + lookup_order, got {names}"
    assert after.get("tool_choice", "auto") == "auto", "verified: let the model decide ('auto')"

_report("ex3", check_ex3)

### Exercise 4 — One output per `call_id`, errors as data

Write `outputs_for(response, handlers)` that returns the list of `function_call_output` input items for **every** `function_call` in `response.output`, in order:

- Each item: `{"type": "function_call_output", "call_id": <the call's call_id>, "output": <JSON string>}`.
- Success: `output` decodes to `{"ok": true, "result": <handler return value>}`.
- Any failure (unknown tool, arguments that aren't valid JSON, the handler raising, a result that `json.dumps` can't encode): `output` decodes to `{"ok": false, "error": <non-empty str>, "message": <non-empty str>}`.
- Never raise. Ignore non-function items (reasoning, messages).

In [ ]:
def outputs_for(response, handlers: dict) -> list[dict]:
    # TODO: one function_call_output per function_call, errors included
    raise NotImplementedError

In [ ]:
def check_ex4():
    def lookup(order_id):
        if order_id not in ORDERS:
            raise LookupError(f"Order {order_id} not found")
        return ORDERS[order_id]

    bad_json = {"type": "function_call", "id": "fc_x4", "call_id": "call_x4", "name": "lookup_order",
                "arguments": "{not json", "status": "completed"}
    resp = fake_response(
        reasoning(),
        fc("lookup_order", {"order_id": "A-1001"}, "call_x1"),
        fc("lookup_order", {"order_id": "A-9999"}, "call_x2"),
        fc("wipe_database", {}, "call_x3"),
        bad_json,
        fc("order_tags", {"order_id": "A-1001"}, "call_x5"),
        msg("(text the model wrote alongside the calls)"),
    )
    out = outputs_for(resp, {"lookup_order": lookup, "order_tags": lambda order_id: {"gift", "fragile"}})  # a set
    assert isinstance(out, list) and len(out) == 5, f"expected 5 outputs (one per function_call), got {len(out) if isinstance(out, list) else out!r}"
    assert [o.get("call_id") for o in out] == ["call_x1", "call_x2", "call_x3", "call_x4", "call_x5"], \
        "use each call's call_id (not its id), in order"
    assert all(o.get("type") == "function_call_output" for o in out), "type must be 'function_call_output'"
    assert all(isinstance(o.get("output"), str) for o in out), "output must be a JSON *string* (json.dumps)"
    decoded = [json.loads(o["output"]) for o in out]
    assert decoded[0].get("ok") is True and decoded[0].get("result", {}).get("order_id") == "A-1001", \
        "success should be {'ok': true, 'result': ...}"
    for i, why in ((1, "handler raised"), (2, "unknown tool"), (3, "invalid JSON arguments"),
                   (4, "result is not JSON-serializable")):
        d = decoded[i]
        assert d.get("ok") is False, f"{why}: ok must be false"
        assert isinstance(d.get("error"), str) and d["error"], f"{why}: give a non-empty 'error' code"
        assert isinstance(d.get("message"), str) and d["message"], f"{why}: give a non-empty 'message'"

_report("ex4", check_ex4)

### Exercise 5 — The bounded Responses loop

Write `run_loop(client, user_text, tools, handlers, max_turns=5)`:

- Start with `input_list = [{"role": "user", "content": user_text}]` and call `client.responses.create(model=MODEL, tools=tools, input=input_list)` each turn.
- If `response.status != "completed"`, raise `RuntimeError` **before** running any tool.
- If the message contains a `refusal` content part, return `"[refused] " + <refusal text>` without running any tool (a refusal is `completed` too, and its `output_text` is empty).
- Append **every** item of `response.output` to `input_list`.
- No `function_call` items → return `response.output_text`.
- Otherwise run each call's handler and append one `function_call_output` per call, in the exercise 4 format (`{"ok": true, "result": ...}` or an error object; you may reuse `outputs_for`), then loop.
- After `max_turns` requests without a final answer, raise `RuntimeError`.

The checker drives your loop with `ScriptedClient`, so no API key is needed.

In [ ]:
def run_loop(client, user_text: str, tools: list[dict], handlers: dict, max_turns: int = 5) -> str:
    # TODO: the bounded tool loop
    raise NotImplementedError

In [ ]:
def check_ex5():
    calls = []
    def lookup(order_id):
        calls.append(order_id)
        return ORDERS[order_id]
    handlers = {"lookup_order": lookup}

    # A: one call, then a final answer
    sc = ScriptedClient([fake_response(reasoning(), fc("lookup_order", {"order_id": "A-1001"}, "call_a")),
                         fake_response(msg("Delivered."))])
    answer = run_loop(sc, "Where is A-1001?", [lookup_order_tool], handlers)
    assert answer == "Delivered.", f"return response.output_text of the final response, got {answer!r}"
    assert len(sc.requests) == 2, f"expected 2 requests, got {len(sc.requests)}"
    assert all(r.get("model") == MODEL and r.get("tools") == [lookup_order_tool] for r in sc.requests), \
        "pass model=MODEL and the tools on every request"
    second = sc.requests[1]["input"]
    types = [item_type(x) for x in second]
    assert "reasoning" in types, "replay EVERY output item, including reasoning items"
    assert "function_call" in types, "replay the function_call item itself"
    outs = [x for x in second if item_type(x) == "function_call_output"]
    assert len(outs) == 1 and outs[0].get("call_id") == "call_a", "send one function_call_output with the call's call_id"
    assert isinstance(outs[0].get("output"), str), "output must be a string"
    assert types.index("function_call") < types.index("function_call_output"), "outputs come after the replayed call"
    assert calls == ["A-1001"], f"run the handler with the call's arguments (expected one lookup of A-1001, got {calls})"
    assert json.loads(outs[0]["output"]) == {"ok": True, "result": ORDERS["A-1001"]}, \
        "the output must carry the handler's result: {'ok': true, 'result': ...}"

    # A2: two calls in one turn -> both handlers run, one output per call_id, in order
    calls.clear()
    sc = ScriptedClient([fake_response(fc("lookup_order", {"order_id": "A-1001"}, "call_p1"),
                                       fc("lookup_order", {"order_id": "A-1002"}, "call_p2")),
                         fake_response(msg("Both found."))])
    run_loop(sc, "Both orders?", [lookup_order_tool], handlers)
    assert calls == ["A-1001", "A-1002"], f"run every call in the turn (got lookups {calls})"
    outs = [x for x in sc.requests[1]["input"] if item_type(x) == "function_call_output"]
    assert [o.get("call_id") for o in outs] == ["call_p1", "call_p2"], "one output per call_id, in order"
    assert [json.loads(o["output"]).get("result", {}).get("order_id") for o in outs] == ["A-1001", "A-1002"], \
        "each output must carry its own call's result"

    # R: refusal -> surfaced, not swallowed as ""
    calls.clear()
    sc = ScriptedClient([fake_response(refusal("I can't help with that."))])
    answer = run_loop(sc, "refuse me", [lookup_order_tool], handlers)
    assert answer == "[refused] I can't help with that.", f"surface refusals as '[refused] <text>', got {answer!r}"
    assert not calls, "never run tools on a refused turn"

    # B: the model never stops -> bounded
    sc = ScriptedClient([fake_response(fc("lookup_order", {"order_id": "A-1001"}, f"call_{i}")) for i in range(10)])
    try:
        run_loop(sc, "loop forever", [lookup_order_tool], handlers, max_turns=3)
    except RuntimeError:
        pass
    else:
        raise AssertionError("raise RuntimeError when max_turns is exceeded")
    assert len(sc.requests) == 3, f"max_turns=3 should allow exactly 3 requests, got {len(sc.requests)}"

    # C: truncated response -> no tool execution
    calls.clear()
    sc = ScriptedClient([fake_response(fc("lookup_order", {"order_id": "A-1001"}, "call_t"),
                                       status="incomplete", incomplete_reason="max_output_tokens")])
    try:
        run_loop(sc, "truncated", [lookup_order_tool], handlers)
    except RuntimeError:
        pass
    else:
        raise AssertionError("raise RuntimeError when status != 'completed'")
    assert not calls, "never execute tools from an incomplete response"

_report("ex5", check_ex5)

### Exercise 6 — Refusal, truncation, then semantics

Write `check_invoice(response)` for a Structured Outputs response whose schema is `InvoiceExtraction` (defined in the setup cell). Return a tuple:

| Situation | Return |
|---|---|
| `status == "incomplete"` | `("incomplete", response.incomplete_details.reason)` |
| The message has a `refusal` part | `("refusal", <refusal text>)` |
| Parsed, but semantic rules fail | `("invalid", [<one or more error strings>])` |
| Parsed and valid | `("ok", <InvoiceExtraction instance>)` |

Parse the text of the **final-answer** message only: skip assistant messages whose `phase` is `"commentary"` (a preamble); a message with no `phase` counts as a final answer. Don't parse `response.output_text`, which joins the text of every message.

Semantic rules: (1) line items sum to `stated_total_usd` within one cent: compare integer cents (`round(x * 100)`), not raw floats; (2) no negative `amount_usd`. Check in this order: incomplete → refusal → parse → semantics.

In [ ]:
def check_invoice(response) -> tuple:
    # TODO: incomplete -> refusal -> parse -> semantic checks
    raise NotImplementedError

In [ ]:
def check_ex6():
    def inv(items, total):
        return json.dumps({"invoice_number": "INV-1",
                           "line_items": [{"description": d, "amount_usd": a} for d, a in items],
                           "stated_total_usd": total})

    kind, val = check_invoice(fake_response(msg(inv([("Keyboard", 120.0), ("Mouse", 29.9)], 149.9))))
    assert kind == "ok", f"valid invoice should be 'ok', got {kind!r}"
    assert isinstance(val, InvoiceExtraction), "return the parsed InvoiceExtraction instance"

    kind, val = check_invoice(fake_response(msg(inv([("Keyboard", 120.0), ("Mouse", 2.99)], 149.9))))
    assert kind == "invalid" and isinstance(val, list) and val and all(isinstance(e, str) for e in val), \
        "a sum mismatch should return ('invalid', [error strings])"

    kind, val = check_invoice(fake_response(msg(inv([("Keyboard", 100.0), ("Discount", -10.0)], 90.0))))
    assert kind == "invalid", "a negative amount should be 'invalid' even when the sum matches"

    kind, val = check_invoice(fake_response(msg("Extracting the invoice now.", phase="commentary"),
                                            msg(inv([("Keyboard", 120.0), ("Mouse", 29.9)], 149.9), phase="final_answer")))
    assert kind == "ok", f"parse only the final_answer message, not the commentary preamble (got {kind!r})"

    kind, val = check_invoice(fake_response(msg(inv([("A", 60.0), ("B", 40.01)], 100.0))))
    assert kind == "ok", "a difference of exactly one cent is allowed (compare integer cents, not floats)"
    kind, val = check_invoice(fake_response(msg(inv([("A", 60.0), ("B", 40.02)], 100.0))))
    assert kind == "invalid", "a two-cent difference is invalid"

    kind, val = check_invoice(fake_response(refusal("I can't help with that.")))
    assert (kind, val) == ("refusal", "I can't help with that."), "a refusal part should return ('refusal', text), status is still 'completed'"

    kind, val = check_invoice(fake_response(msg('{"invoice_number": "INV'), status="incomplete",
                                            incomplete_reason="max_output_tokens"))
    assert (kind, val) == ("incomplete", "max_output_tokens"), "check status before parsing a truncated payload"

_report("ex6", check_ex6)

### Bonus (live, not graded, ~$0.01) — your loop against the real API

Runs your `run_loop` from exercise 5 with the real client and the fake order data.

In [ ]:
# (live) bonus: not graded
try:
    print(run_loop(client, "What's the status of order A-1002?", [lookup_order_tool],
                   {"lookup_order": lambda order_id: ORDERS.get(order_id) or {"error": "not_found"}}))
except NotImplementedError:
    print("Implement run_loop (exercise 5) first.")

---
## Part C — Architecture scenario

**Scenario.** A regional airline wants a disruption assistant built on the Responses API. During delays it must: read flight status and passenger bookings (internal APIs, about 30 endpoints exist), rebook passengers onto other flights (a write), issue meal or travel vouchers up to 600 EUR (a write with money attached), and file a structured case summary into the CRM after each conversation. Peak load is about 2,000 conversations per hour, and the booking backend times out a few percent of the time. Legal wants proof that no voucher was issued outside policy.

**Your task (write 10–20 bullet points):** design the function-calling layer. Cover the tool surface and descriptions, schemas and strict mode, `tool_choice` / `allowed_tools` / parallel calls per phase, the error contract in `function_call_output`, how the CRM summary is produced and validated, where policy is enforced, loop limits and state, and model choice.

*Your answer:*

- ...

<details><summary>Rubric: what a strong answer contains (open after writing yours)</summary>

- **Small, role-scoped tool surface.** Don't expose 30 endpoints: a handful of task-level functions (`get_flight_status`, `get_booking`, `search_alternatives`, `rebook_passenger`, `issue_voucher`), with the rest deferred via tool search or left out. Descriptions say what each returns, when to use it vs. its neighbours, ID formats, and side effects.
- **Offload known arguments to code.** The passenger's booking reference comes from the authenticated session, not from the model; write tools don't take an identity the model could change.
- **`strict: true` explicitly** on every function; required + nullable for optional fields; enums for voucher type and reason codes (with `other` + detail).
- **Phase gating with a stable tool list.** `allowed_tools` (read-only, `mode: "required"` while gathering facts; write tools only after the passenger and flight are confirmed). `parallel_tool_calls: false` during write phases so rebook and voucher run one per turn (zero or one call per turn); that serializes the calls but doesn't choose their order, so the voucher handler checks that the rebooking exists (or `allowed_tools` exposes one write tool per step). Parallel reads are fine.
- **Policy in code, not prompts.** The `issue_voucher` handler checks eligibility (delay length, cabin, previous vouchers), caps the amount at 600 EUR, is idempotent (one voucher per passenger per disruption via an idempotency key), and logs an audit record. Above a threshold or on an edge case: return `requires_human_approval`. This gives Legal its proof.
- **Error contract.** Every `call_id` gets a `function_call_output`; errors are JSON (`ok: false`, `error` code, `retryable`, a user-safe `message`, next step). Timeouts are retried with backoff **inside** the tool; validation errors return immediately; internal details are not leaked.
- **Loop control.** Driven by `function_call` items and `status`; never act on `incomplete` or refused turns; own `max_turns` (and a timeout) with human handoff; `max_tool_calls` doesn't bound functions.
- **CRM summary via Structured Outputs** (`responses.parse(text_format=CaseSummary)` or `text.format`), not a fake tool: required + nullable fields, `outcome` enum incl. `unclear`. Check refusals / `incomplete` first, then **semantic** validation against the system of record (voucher IDs and amounts in the summary must match what the handlers actually issued; computed totals in code).
- **State and data.** Manual replay of every output item (reasoning included) or `previous_response_id`; consider `store` / retention for passenger PII, and send only high-signal fields back in tool outputs.
- **Model choice and evals.** Start with `gpt-6-luna` for cost at 2,000 conversations/hour; measure tool-selection accuracy and policy violations on a test set; move hard cases (or the whole flow) to `gpt-6.1-sol` only if the evals justify it.
- **Observability.** Log every call/output pair with `call_id`, latency, error codes and approvals for audits.

Scoring guide: 8+ of these points well argued = strong; 5–7 = adequate; fewer = re-read README sections 2.2–2.8.
</details>

---
## Scorecard

In [ ]:
N_EXERCISES = 6
quiz = globals().get("QUIZ_SCORE", 0)
passed = sum(RESULTS.get(f"ex{i}", False) for i in range(1, N_EXERCISES + 1))
overall = (quiz + passed) / (10 + N_EXERCISES)
print(f"Quiz:      {quiz}/10")
print(f"Exercises: {passed}/{N_EXERCISES}")
print(f"Overall:   {overall:.0%}  ->  {'PASS' if overall >= 0.72 else 'RETRY'} (pass mark 72%)")
print("Part C: self-graded with the rubric above.")

Next chapter: [Chapter 3 — Agents SDK and Agents API](../03-agents-sdk-and-agents-api/README.md), where the loop you just wrote is packaged by the Agents SDK `Runner`.

---
## Solutions (spoilers)

<details><summary>Part A — quiz answers with explanations</summary>

- **Q1 (Round trip): C.** Results go back as a `function_call_output` input item linked by `call_id` (`call_...`). A is the Chat Completions tool message, B uses the item `id` (`fc_...`), D is Claude's `tool_result` shape.
- **Q2 (Arguments): B.** `arguments` is always a JSON string in the Responses API; strict mode guarantees its *content* matches the schema, not its Python type. `input` is Claude's field (C).
- **Q3 (Strict schema): A.** OpenAI strict mode requires every property in `required` and `additionalProperties: false`; optional values are expressed as a union with `null`. B breaks the other rule, C gives up the guarantee, D doesn't satisfy the `required` rule.
- **Q4 (Strict default): D.** Per the function-calling guide, omitting `strict` in Responses attempts strict mode and falls back silently when the schema isn't compatible. Set `strict: true` explicitly so incompatible schemas fail loudly.
- **Q5 (tool_choice): B.** `required` = call one or more functions, model's choice. A may answer in text, C forces the invoice schema onto every document, D forbids calls.
- **Q6 (allowed_tools): C.** `allowed_tools` restricts the callable subset without changing the `tools` list, which the docs recommend for prompt-caching savings. A is a prompt rule for a security property; B works but changes the cached prefix; D only applies to built-in tools.
- **Q7 (Parallel calls): A.** `parallel_tool_calls: false` (top-level) ensures zero or one tool call per turn. C is Claude's field name, not OpenAI's; B forbids tools; D only counts built-in tools.
- **Q8 (Refusals): D.** A refusal comes back with `status: completed` and a `refusal` part that doesn't follow the schema, so there is nothing to parse. B would show `status: incomplete`; A would be rejected at request time; C would raise a client error.
- **Q9 (Migration): B.** Structured Outputs moved from `response_format` to `text.format`, with `name`/`schema` directly inside `format`. C raises a `TypeError` in the SDK; D is JSON mode (no schema adherence).
- **Q10 (Tool errors): A.** Every `call_id` needs an output, and errors are data the model can act on. `function_call_output` has no `is_error` field (that's Claude). Retrying a not-found error never helps.

</details>

<details><summary>Exercise 1 — reference solution</summary>

```python
def to_responses_tool(chat_tool: dict) -> dict:
    if chat_tool.get("type") != "function" or "function" not in chat_tool:
        raise ValueError("Expected a Chat Completions function tool: {'type': 'function', 'function': {...}}")
    fn = chat_tool["function"]
    tool = {"type": "function", "name": fn["name"]}
    for key in ("description", "parameters", "strict"):
        if key in fn:
            tool[key] = copy.deepcopy(fn[key])
    return tool
```

</details>

<details><summary>Exercise 2 — reference solution</summary>

```python
def make_strict(schema: dict) -> dict:
    result = copy.deepcopy(schema)

    def is_object(node: dict) -> bool:
        t = node.get("type")
        return t == "object" or (isinstance(t, list) and "object" in t)

    def make_nullable(node: dict) -> None:
        t = node.get("type")
        if isinstance(t, str):
            if t != "null":
                node["type"] = [t, "null"]
        elif isinstance(t, list):
            if "null" not in t:
                node["type"] = t + ["null"]
        elif "anyOf" in node:
            if {"type": "null"} not in node["anyOf"]:
                node["anyOf"].append({"type": "null"})
        elif "$ref" in node:
            node["anyOf"] = [{"$ref": node.pop("$ref")}, {"type": "null"}]
        if "enum" in node and None not in node["enum"]:
            node["enum"].append(None)          # otherwise the enum still rules out null

    def fix(node: dict) -> None:
        if is_object(node):
            props = node.get("properties", {})
            originally_required = set(node.get("required", []))
            for name, sub in props.items():
                fix(sub)
                if name not in originally_required:
                    make_nullable(sub)
            node["required"] = list(props)
            node["additionalProperties"] = False
        if isinstance(node.get("items"), dict):
            fix(node["items"])
        for sub in node.get("anyOf", []):
            fix(sub)
        for sub in node.get("$defs", {}).values():
            fix(sub)

    fix(result)
    return result
```

</details>

<details><summary>Exercise 3 — reference solution</summary>

```python
READ_ONLY_TOOLS = [{"type": "function", "name": "get_customer"},
                   {"type": "function", "name": "lookup_order"}]

def build_turn_request(verified: bool, input_list: list) -> dict:
    tool_choice = "auto" if verified else {"type": "allowed_tools", "mode": "required", "tools": READ_ONLY_TOOLS}
    return {
        "model": MODEL,
        "instructions": SUPPORT_INSTRUCTIONS,
        "tools": SUPPORT_TOOLS,
        "tool_choice": tool_choice,
        "parallel_tool_calls": False,
        "input": input_list,
    }
```

</details>

<details><summary>Exercise 4 — reference solution</summary>

```python
def outputs_for(response, handlers: dict) -> list[dict]:
    outputs = []
    for item in response.output:
        if item.type != "function_call":
            continue
        handler = handlers.get(item.name)
        if handler is None:
            payload = {"ok": False, "error": "unknown_tool", "message": f"No tool named {item.name!r}."}
        else:
            try:
                payload = {"ok": True, "result": handler(**json.loads(item.arguments))}
            except json.JSONDecodeError as exc:
                payload = {"ok": False, "error": "invalid_json", "message": str(exc)}
            except Exception as exc:
                payload = {"ok": False, "error": type(exc).__name__, "message": str(exc) or "failed"}
        try:
            output = json.dumps(payload)
        except (TypeError, ValueError):      # the handler returned something JSON can't encode
            output = json.dumps({"ok": False, "error": "internal_error", "message": "Result could not be encoded."})
        outputs.append({"type": "function_call_output", "call_id": item.call_id, "output": output})
    return outputs
```

</details>

<details><summary>Exercise 5 — reference solution</summary>

```python
def run_loop(client, user_text: str, tools: list[dict], handlers: dict, max_turns: int = 5) -> str:
    input_list: list = [{"role": "user", "content": user_text}]
    for _ in range(max_turns):
        response = client.responses.create(model=MODEL, tools=tools, input=input_list)
        if response.status != "completed":
            raise RuntimeError(f"Response {response.status}: {response.incomplete_details}")
        refusals = [part.refusal for item in response.output if item.type == "message"
                    for part in item.content if part.type == "refusal"]
        if refusals:
            return f"[refused] {refusals[0]}"
        input_list += response.output
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response.output_text
        input_list += outputs_for(response, handlers)   # exercise 4: one output per call_id, never raises
    raise RuntimeError(f"No final answer after {max_turns} turns")
```

</details>

<details><summary>Exercise 6 — reference solution</summary>

```python
def check_invoice(response) -> tuple:
    if response.status == "incomplete":
        return ("incomplete", response.incomplete_details.reason)
    final = [item for item in response.output
             if item.type == "message" and item.phase in (None, "final_answer")]   # skip commentary preambles
    for item in final:
        for part in item.content:
            if part.type == "refusal":
                return ("refusal", part.refusal)
    text = next(part.text for item in final for part in item.content if part.type == "output_text")
    invoice = InvoiceExtraction.model_validate_json(text)
    errors = []
    computed = sum(round(i.amount_usd * 100) for i in invoice.line_items)        # integer cents
    if abs(computed - round(invoice.stated_total_usd * 100)) > 1:
        errors.append(f"line items sum to {computed / 100:.2f} but stated_total_usd is {invoice.stated_total_usd}")
    negatives = [i.description for i in invoice.line_items if i.amount_usd < 0]
    if negatives:
        errors.append(f"negative amounts on: {negatives}")
    return ("invalid", errors) if errors else ("ok", invoice)
```

</details>